In [117]:
# Install openrouteservice for first time users
# !pip install openrouteservice

In [ ]:
import openrouteservice
import pandas as pd
import numpy as np
from tqdm import tqdm

In [118]:
# Load in rental data
rental = pd.read_csv("rea_data/domain/Data/vic_rentals_all_cleaned.csv")
rental_coords = rental[["listing_id", "lon","lat"]]
rental_coords.head()

,listing_id,lon,lat
0,16782629,144.87091,-37.830982
1,17471867,144.86755,-37.826218
2,17721851,144.86632,-37.831226
3,17725855,144.86768,-37.827423
4,17745057,144.86790,-37.826270


In [ ]:
# Load in train stops data
train = pd.read_csv("data\metro_train_stops.txt")
station_coords = train[["stop_id", "stop_lon","stop_lat"]]
station_coords.head()

,stop_id,stop_lon,stop_lat
0,10117,145.112473,-37.873763
1,10920,144.956043,-37.811880
2,10921,144.955968,-37.811725
3,10922,144.962513,-37.809973
4,10923,144.962505,-37.809865


In [ ]:
# Define Haversine distance function
def haversine(lon1, lat1, lon2, lat2):
    """
    Compute distance (meters) between two points using Haversine formula
    """
    earth_mean_radius = 6371000
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi1 = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    a = np.sin(dphi1/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return earth_mean_radius * c

In [122]:
# Find top 3 closest train stops for each rental
closest_3 = []
for r_lon, r_lat in rental_coords[["lon", "lat"]].values:
    distances = [haversine(r_lon, r_lat, s_lon, s_lat) for s_lon , s_lat in station_coords[["stop_lon", "stop_lat"]].values]
    indices = np.argsort(distances)[:3]
    closest_3.append(indices)

rental_coords["top3_idx"] = closest_3
rental_coords["top3_tuple"] = rental_coords["top3_idx"].apply(lambda x: tuple(sorted(x)))

C:\Users\chinj\AppData\Local\Temp\ipykernel_10832\3355039345.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rental_coords["top3_idx"] = closest_3
C:\Users\chinj\AppData\Local\Temp\ipykernel_10832\3355039345.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rental_coords["top3_tuple"] = rental_coords["top3_idx"].apply(lambda x: tuple(sorted(x)))


In [ ]:
# Group results by their top 3 closest train stops
groups = rental_coords.groupby("top3_tuple")["listing_id"].apply(list).reset_index()
groups["num_rentals"] = groups["listing_id"].apply(len)

In [ ]:
# API key for OpenRouteService (ORS); hidden here for privacy
API_KEY = NA # Replace NA with your actual key when running

# Initialize ORS client with the API key
client = openrouteservice.Client(key = API_KEY)

In [124]:
# Batch rentals to stay under API route limit
top_n_stations = 3
max_route_per_batch = 3500

batches = []
current_batch = []
current_batch_rentals = 0

# Loop over each group of rentals that share the same top 3 train stops
for _, row in groups.iterrows():
    group_name = row["top3_tuple"]
    rentals = row["listing_id"]
    group_size = len(rentals)

    start = 0
    # Split group into chunks so that adding them won't exceed ORS route limit
    while start < group_size:
        # Estimate number of rentals allowed to add to current batch without exceeding ORS route limit
        num_groups_if_added = len(current_batch) + 1
        max_rentals_for_this_chunk = max(
            (max_route_per_batch // (num_groups_if_added * top_n_stations)) - current_batch_rentals,
            1
        )

        # Determine slice of rentals for this chunk
        end = min(start + max_rentals_for_this_chunk, group_size)
        chunk = rentals[start:end]

        # Compute estimated number of routes if this chunk is added
        num_origins = current_batch_rentals + len(chunk)
        num_destinations = num_groups_if_added * top_n_stations
        estimated_routes = num_origins * num_destinations

        # Start new batch if adding this chunk exceeds ORS route limit
        if estimated_routes >= max_route_per_batch: 
            if current_batch:
                batches.append(current_batch)
            current_batch = [(group_name, chunk)]     
            current_batch_rentals = len(chunk)
        else:
            # Add chunk to batch otherwise
            current_batch.append((group_name, chunk))
            current_batch_rentals += len(chunk)
        start = end

# Append last batch if there are any remaining rentals
if current_batch:
        batches.append(current_batch)

print(f"Created {len(batches)} batches with full ORS route limit logic.")

Created 94 batches with full ORS route limit logic.


In [ ]:
# Print and check summary of batches
for i, batch in enumerate(batches):
    num_groups = len(batch)
    num_rentals = sum(len(rentals) for _, rentals in batch)
    print(f"Batch {i+1}: {num_groups} groups, {num_rentals} rentals")

Batch 1: 9 groups, 124 rentals
Batch 2: 13 groups, 88 rentals
Batch 3: 14 groups, 79 rentals
Batch 4: 8 groups, 145 rentals
Batch 5: 14 groups, 83 rentals
Batch 6: 14 groups, 83 rentals
Batch 7: 19 groups, 61 rentals
Batch 8: 15 groups, 77 rentals
Batch 9: 9 groups, 125 rentals
Batch 10: 9 groups, 129 rentals
Batch 11: 7 groups, 166 rentals
Batch 12: 9 groups, 129 rentals
Batch 13: 7 groups, 166 rentals
Batch 14: 6 groups, 189 rentals
Batch 15: 9 groups, 129 rentals
Batch 16: 8 groups, 141 rentals
Batch 17: 12 groups, 97 rentals
Batch 18: 12 groups, 97 rentals
Batch 19: 8 groups, 145 rentals
Batch 20: 8 groups, 133 rentals
Batch 21: 11 groups, 106 rentals
Batch 22: 11 groups, 106 rentals
Batch 23: 11 groups, 106 rentals
Batch 24: 10 groups, 116 rentals
Batch 25: 10 groups, 116 rentals
Batch 26: 14 groups, 80 rentals
Batch 27: 9 groups, 129 rentals
Batch 28: 4 groups, 291 rentals
Batch 29: 4 groups, 291 rentals
Batch 30: 4 groups, 291 rentals
Batch 31: 7 groups, 166 rentals
Batch 32: 4 

In [ ]:
# Call ORS to get distance to closest train stops
all_rental_ids = []
all_distances = []
all_station_ids = []

for batch in tqdm(batches, desc="Processing batches"):
    batch_rentals = [r for _, rentals in batch for r in rentals]
    batch_coords = rental_coords.set_index("listing_id").loc[batch_rentals][["lon", "lat"]].values.tolist()

    batch_station_indices = [
        rental_coords.loc[rental_coords["listing_id"]==r, "top3_idx"].values[0] for r in batch_rentals
    ]

    unique_station_indices = list(set(idx for indices in batch_station_indices for idx in indices))

    origins = batch_coords
    destinations = station_coords.loc[station_coords.index.isin(unique_station_indices)][["stop_lon", "stop_lat"]].values.tolist()
    dest_ids_map = {idx: station_coords["stop_id"].iloc[idx] for idx in unique_station_indices}
   
    station_pos_map = {idx: i for i, idx in enumerate(unique_station_indices)}

    num_origins = len(origins)
    num_destinations = len(destinations)
    num_routes = num_origins * num_destinations
    print(f"Checking batch → {num_origins} origins × {num_destinations} destinations = {num_routes} routes")

    # Call ORS distance matrix
    matrix = client.distance_matrix(
        locations = origins + destinations,
        profile = "driving-car",
        metrics = ["distance"],
        sources = list(range(len(origins))),
        destinations = list(range(len(origins), len(origins)+len(destinations)))
    )

    # Extract distances and train stop IDs for top 3 train stops
    for i, rental in enumerate(batch_rentals):
        top_indices = batch_station_indices[i]
        dest_positions = [station_pos_map[idx] for idx in top_indices]
        rental_distances = [matrix["distances"][i][pos] for pos in dest_positions]
        rental_station_ids = [dest_ids_map[idx] for idx in top_indices]

        sorted_idx = np.argsort(rental_distances)
        top_n_distances = [rental_distances[k] for k in sorted_idx]
        top_n_ids = [rental_station_ids[k] for k in sorted_idx]

        all_rental_ids.append(batch_rentals[i])
        all_distances.append(top_n_distances)
        all_station_ids.append(top_n_ids)

Processing batches:   0%|          | 0/94 [00:00<?, ?it/s]

Checking batch → 124 origins × 11 destinations = 1364 routes


Processing batches:   1%|          | 1/94 [00:02<03:11,  2.06s/it]

Checking batch → 88 origins × 22 destinations = 1936 routes


Processing batches:   2%|▏         | 2/94 [00:02<01:40,  1.09s/it]

...

Processing batches:  98%|█████████▊| 92/94 [03:49<00:01,  1.70it/s]

Checking batch → 49 origins × 37 destinations = 1813 routes


Processing batches:  99%|█████████▉| 93/94 [03:49<00:00,  1.86it/s]

Checking batch → 61 origins × 29 destinations = 1769 routes


Processing batches: 100%|██████████| 94/94 [03:50<00:00,  2.45s/it]


In [ ]:
# Save ORS distances for train stops into dataframe
df = pd.DataFrame({
    "rental_id": all_rental_ids,
    "top_distances": all_distances,
    "top_station_ids": all_station_ids
})

df.head()

,rental_id,top_distances,top_station_ids
0,17694443,"[3387.1, 22581.13, 22631.29]","[26199, 11219, 10117]"
1,14799583,"[20639.18, 20689.34, 21928.44]","[11219, 10117, vic:rail:JOR]"
2,16807603,"[19489.88, 19540.04, 20779.13]","[11219, 10117, vic:rail:JOR]"
3,17744784,"[20174.74, 20224.9, 21464.0]","[11219, 10117, vic:rail:JOR]"
4,17745926,"[18491.52, 18541.69, 19780.78]","[11219, 10117, vic:rail:JOR]"


In [ ]:
# Call ORS to get distance to Melbourne CBD
melbourne_cbd = [144.96246406706035, -37.81227768714442]

max_route_per_batch = 3000

# Batch rentals for CBD calculation
batches = [rental_coords["listing_id"][i:i+max_route_per_batch]
           for i in range(0, len(rental_coords), max_route_per_batch)]

all_rental_ids = []
all_distance_to_cbd = []

for batch in tqdm(batches, desc="Calculating ORS distance to CBD in batches"):
    batch_coords = rental_coords.set_index("listing_id").loc[batch][["lon", "lat"]].values.tolist()

    num_origins = len(batch_coords)
    num_destinations = 1
    num_routes = num_origins * num_destinations
    print(f"Checking batch → {num_origins} origins × {num_destinations} destinations = {num_routes} routes")

    # Call ORS distance matrix
    matrix = client.distance_matrix(
        locations = batch_coords + [melbourne_cbd],
        profile = "driving-car",
        metrics = ["distance"],
        sources=list(range(len(batch_coords))),
        destinations=[len(batch_coords)]
    )

    # Extract distances for CBD 
    for i, rental_id in enumerate(batch):
        all_rental_ids.append(rental_id)
        all_distance_to_cbd.append(matrix["distances"][i][0])

Calculating ORS distance to CBD in batches:   0%|          | 0/4 [00:00<?, ?it/s]

Checking batch → 3000 origins × 1 destinations = 3000 routes


Calculating ORS distance to CBD in batches:  25%|██▌       | 1/4 [00:04<00:12,  4.05s/it]

Checking batch → 3000 origins × 1 destinations = 3000 routes


Calculating ORS distance to CBD in batches:  50%|█████     | 2/4 [00:05<00:04,  2.30s/it]

Checking batch → 3000 origins × 1 destinations = 3000 routes


Calculating ORS distance to CBD in batches:  75%|███████▌  | 3/4 [00:07<00:02,  2.13s/it]

Checking batch → 2618 origins × 1 destinations = 2618 routes


Calculating ORS distance to CBD in batches: 100%|██████████| 4/4 [00:08<00:00,  2.02s/it]


In [ ]:
# Save ORS distances for CBD to dataframe
df_cbd = pd.DataFrame({
    "rental_id": all_rental_ids,
    "distance_to_cbd": all_distance_to_cbd
})

df_cbd.head()

,rental_id,distance_to_cbd
0,16782629,10681.23
1,17471867,11022.42
2,17721851,11070.31
3,17725855,10739.04
4,17745057,10941.09


In [ ]:
# Combine train stop distances and CBD distances
combined_df = df.merge(
    df_cbd,
    on="rental_id",
    how="inner"
)

combined_df.head()

,rental_id,top_distances,top_station_ids,distance_to_cbd
0,17694443,"[3387.1, 22581.13, 22631.29]","[26199, 11219, 10117]",21520.46
1,14799583,"[20639.18, 20689.34, 21928.44]","[11219, 10117, vic:rail:JOR]",19578.50
2,16807603,"[19489.88, 19540.04, 20779.13]","[11219, 10117, vic:rail:JOR]",18429.20
3,17744784,"[20174.74, 20224.9, 21464.0]","[11219, 10117, vic:rail:JOR]",19114.06
4,17745926,"[18491.52, 18541.69, 19780.78]","[11219, 10117, vic:rail:JOR]",17430.85


In [ ]:
# Save final CSV
combined_df.to_csv("rentals_distances_to_train_stops_and_cbd.csv", index=False)